In [2]:
!pip install pandas numpy scikit-learn mord openpyxl

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 40.1 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 39.8 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 35.6 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 30.5 MB/s  0:00:01m0:00:0100:01
  Created wheel for mord: filename=mord-0.7-py3-none-any.whl size=9953 sha256=d95efdc472f2a7ce42753790bed9f0068c65a35ceb09afc8cd4abd2a81efb0ff
  Stored in directory: /home/codespace/.cache/pip/wheels/80/3e/3b/13f1adf346cad0fec675db328e4b0d814795c6c8e2fb659122
Successfully built mord
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
file_path = "./Post Methods Sheet.csv"

In [5]:
import os

print(os.getcwd())

for root, dirs, files in os.walk("."):
    if "Post Methods Sheet.csv" in files:
        print(os.path.join(root, "Post Methods Sheet.csv"))

/workspaces/BSAN-Project-2
./Post Methods Sheet.csv


In [8]:
%pip install pandas numpy matplotlib scikit-learn mord openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 30.3 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 49.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 28.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 37.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [matplotlib]7 [matplotlib]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

from mord import LogisticAT


# =========================
# 1. LOAD DATA
# =========================

file_path = "./Post Methods Sheet.csv"
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)


# =========================
# 2. CREATE POSITION GROUPS
# =========================

position_map = {
    "QB": "Skill", "RB": "Skill", "FB": "Skill", "WR": "Skill", "TE": "Skill",
    "OT": "OL", "OG": "OL", "G": "OL", "C": "OL", "OC": "OL", "IOL": "OL", "OL": "OL",
    "EDGE": "DL", "DE": "DL", "DT": "DL", "DL": "DL",
    "LB": "LB", "ILB": "LB", "OLB": "LB",
    "CB": "Secondary", "S": "Secondary", "SAF": "Secondary", "DB": "Secondary",
    "K": "Special Teams", "P": "Special Teams", "LS": "Special Teams"
}

df["Position Group"] = df["Position"].map(position_map).fillna(df["Position"])


# =========================
# 3. CREATE ORDINAL DRAFT TIER
# =========================

def draft_tier(pick):
    if pick <= 32:
        return "Round 1"
    elif pick <= 96:
        return "Rounds 2-3"
    elif pick <= 160:
        return "Rounds 4-5"
    elif pick <= 262:
        return "Rounds 6-7"
    else:
        return "Undrafted"

df["Draft Tier"] = df["Overall Pick"].apply(draft_tier)

labels_order = [
    "Round 1",
    "Rounds 2-3",
    "Rounds 4-5",
    "Rounds 6-7",
    "Undrafted"
]

tier_to_num = {
    "Round 1": 0,
    "Rounds 2-3": 1,
    "Rounds 4-5": 2,
    "Rounds 6-7": 3,
    "Undrafted": 4
}

num_to_tier = {
    0: "Round 1",
    1: "Rounds 2-3",
    2: "Rounds 4-5",
    3: "Rounds 6-7",
    4: "Undrafted"
}

df["Draft Tier Ordinal"] = df["Draft Tier"].map(tier_to_num)

print("\nDraft Tier Counts:")
print(df["Draft Tier"].value_counts())


# =========================
# 4. FEATURES
# ELO, School, and Arm Length excluded
# =========================

features = [
    "Season",
    "Position",
    "Position Group",
    "Conference",
    "Height (inches)",
    "Weight(lbs)",
    "Hand (inches)",
    "Wingspan (inches)",
    "40-Yard Dash (seconds)",
    "Vertical (inches)",
    "Bench (reps)",
    "Broad Jump (inches)",
    "3-Cone (seconds)",
    "Shuttle (seconds)",
    "Combine Participation",
    "FPI",
    "Efficiencies Overall",
    "Efficiencies Offense",
    "Efficiencies Defense",
    "Efficiencies SpecialTeams",
    "CFP Appearance",
    "National Award"
]


# =========================
# 5. TRAIN 2023-2025, TEST 2026
# =========================

train_df = df[df["Season"].isin([2023, 2024, 2025])].copy()
test_df = df[df["Season"] == 2026].copy()

print("\nTraining rows:", len(train_df))
print("Testing rows:", len(test_df))

X_train = train_df[features]
y_train = train_df["Draft Tier Ordinal"]

X_test = test_df[features]
y_test = test_df["Draft Tier Ordinal"]


# =========================
# 6. BUILD MODEL
# =========================

def build_model(feature_list):
    categorical_features = [
        col for col in ["Position", "Position Group", "Conference"]
        if col in feature_list
    ]

    numeric_features = [
        col for col in feature_list if col not in categorical_features
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", "passthrough", numeric_features)
        ]
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("ordinal_model", LogisticAT(alpha=1.0))
        ]
    )

    return model


# =========================
# 7. MANUAL REPORT
# =========================

def print_manual_report(y_true, y_pred, title):
    y_true_labels = pd.Series(y_true).map(num_to_tier)
    y_pred_labels = pd.Series(y_pred).map(num_to_tier)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true_labels,
        y_pred_labels,
        labels=labels_order,
        zero_division=0
    )

    print(f"\n{title}")
    print(f"{'Tier':<14}{'Precision':>12}{'Recall':>12}{'F1-Score':>12}{'Support':>12}")

    for label, p, r, f, s in zip(labels_order, precision, recall, f1, support):
        print(f"{label:<14}{p:>12.2f}{r:>12.2f}{f:>12.2f}{s:>12}")

    print("\nAccuracy:", round(accuracy_score(y_true, y_pred), 4))

    ordinal_error = np.mean(np.abs(np.array(y_true) - np.array(y_pred)))
    print("Average Ordinal Error:", round(ordinal_error, 4))


# =========================
# 8. FIT MODEL AND PREDICT 2026
# =========================

overall_model = build_model(features)

overall_model.fit(X_train, y_train)

y_pred = overall_model.predict(X_test)

print_manual_report(
    y_test,
    y_pred,
    "2026 OUT-OF-SAMPLE ORDINAL PREDICTION REPORT"
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


# =========================
# 9. FEATURE COEFFICIENTS
# =========================

ordinal_model = overall_model.named_steps["ordinal_model"]
feature_names = overall_model.named_steps["preprocessor"].get_feature_names_out()

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": ordinal_model.coef_
}).sort_values(by="Coefficient", key=abs, ascending=False)

print("\nTop 20 Features by Absolute Coefficient Size:")
print(importance_df.head(20).to_string(index=False))


# =========================
# 10. SAVE 2026 PLAYER PREDICTIONS
# =========================

results = test_df[
    [
        "Season",
        "Player",
        "Position",
        "Position Group",
        "Conference",
        "Overall Pick"
    ]
].copy()

results["Actual Tier Number"] = y_test.values
results["Predicted Tier Number"] = y_pred

results["Actual Tier"] = results["Actual Tier Number"].map(num_to_tier)
results["Predicted Tier"] = results["Predicted Tier Number"].map(num_to_tier)

results["Correct"] = np.where(
    results["Actual Tier Number"] == results["Predicted Tier Number"],
    "Correct",
    "Incorrect"
)

results["Ordinal Error"] = abs(
    results["Actual Tier Number"] - results["Predicted Tier Number"]
)

results.to_csv(
    "2026_out_of_sample_ordinal_predictions.csv",
    index=False
)

importance_df.to_csv(
    "2026_out_of_sample_feature_coefficients.csv",
    index=False
)

print("\nSaved:")
print("2026_out_of_sample_ordinal_predictions.csv")
print("2026_out_of_sample_feature_coefficients.csv")

print("\nSample Predictions:")
print(results.head(25).to_string(index=False))


# =========================
# 11. CHECK COEFFICIENTS AND THRESHOLDS
# =========================

print("\nNumber of coefficients:", len(ordinal_model.coef_))
print("Number of thresholds:", len(ordinal_model.theta_))

Dataset shape: (1272, 27)

Draft Tier Counts:
Draft Tier
Undrafted     347
Rounds 6-7    298
Rounds 2-3    252
Rounds 4-5    247
Round 1       128
Name: count, dtype: int64

Training rows: 965
Testing rows: 307

2026 OUT-OF-SAMPLE ORDINAL PREDICTION REPORT
Tier             Precision      Recall    F1-Score     Support
Round 1               0.50        0.38        0.43          32
Rounds 2-3            0.32        0.14        0.20          63
Rounds 4-5            0.23        0.33        0.27          61
Rounds 6-7            0.26        0.53        0.35          68
Undrafted             0.37        0.12        0.18          83

Accuracy: 0.2834
Average Ordinal Error: 1.0651

Confusion Matrix:
[[12  8  4  8  0]
 [ 5  9 19 27  3]
 [ 5  3 20 25  8]
 [ 1  4 21 36  6]
 [ 1  4 24 44 10]]

Top 20 Features by Absolute Coefficient Size:
                          Feature  Coefficient
              num__National Award    -2.507714
      num__40-Yard Dash (seconds)     0.692299
                 ca